# Fashion Recommendation Prototype

## 1. Objective

The objective of this notebook is to build a first recommendation prototype using the previously trained style and clothing type classifiers.

The prototype takes an input clothing image and predicts:

- the style of the item
- the clothing type of the item

These predictions are then used to retrieve compatible items from the dataset.

The first version of the recommendation system uses a simple rule-based approach:

- recommend items with the same predicted style
- exclude items with the same predicted clothing type

For example, if the input image is predicted as a gothic jacket, the system should recommend gothic items that are not jackets, such as gothic pants, gothic shoes, or gothic tshirts.

This notebook does not attempt to solve full outfit compatibility yet. Instead, it creates a practical first prototype that connects the classification models to a simple retrieval-based recommendation system.

## 2. Project Context

The previous notebooks developed two baseline models. The first model predicts fashion style, while the second model predicts clothing type.

The style classifier provides information about the visual identity of the item, such as whether it belongs to formal, gothic, sporty, or streetwear fashion. The clothing type classifier identifies the functional category of the item, such as jacket, pants, shoes, or tshirt.

Combining these two predictions makes it possible to build a simple recommendation system. Instead of recommending random items, the system can retrieve items that match the predicted style while avoiding items of the same type.

This approach is intentionally simple. It does not yet use visual embeddings, user preferences, or learned outfit compatibility. However, it is useful as a first prototype because it shows how the trained models can be used in a functional recommendation workflow.

## 3. Recommendation Logic

The recommendation logic in this notebook is based on two predicted labels:

- predicted style
- predicted clothing type

The system follows this rule:

```text
Recommend items where:
    item style = predicted style
    item type != predicted type
```

This means that the system keeps the style consistent while recommending different clothing categories.

For example:
```text
Input image:
    predicted style = gothic
    predicted type = jacket

Recommended items:
    gothic pants
    gothic shoes
    gothic tshirts
```

This creates a basic outfit-building logic where the recommended items are stylistically related but functionally different from the input item.

## 4. Import Libraries

The notebook uses `PyTorch` and torchvision to load the trained models and process input images. It also uses pandas and matplotlib to manage metadata and visualize recommendations.

In [1]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
from torchvision import models, transforms

# Reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using device: CUDA")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using device: MPS")
else:
    device = torch.device("cpu")
    print("Using device: CPU")

Using device: CUDA


## 5. Load Trained Models

The recommendation prototype uses the two previously trained baseline models:

- the style classifier
- the clothing type classifier

Both models use the same `ResNet34` architecture, but each model has a different final classification layer depending on the target labels. The saved checkpoints also include the class names, which are needed to convert model outputs into readable predictions.

In [2]:
style_model_path = "../models/style_resnet34.pth"
type_model_path = "../models/type_resnet34.pth"

style_checkpoint = torch.load(style_model_path, map_location=device)
type_checkpoint = torch.load(type_model_path, map_location=device)

style_class_names = style_checkpoint["class_names"]
type_class_names = type_checkpoint["class_names"]

print("Style classes:", style_class_names)
print("Type classes:", type_class_names)

Style classes: ['formal', 'gothic', 'sporty', 'streetwear']
Type classes: ['jacket', 'pants', 'shoes', 'tshirt']


In [3]:
def load_resnet34_classifier(checkpoint, num_classes):
  model = models.resnet34(weights=None)
  in_features = model.fc.in_features
  model.fc = nn.Linear(in_features, num_classes)
  model.load_state_dict(checkpoint["model_state_dict"])
  model = model.to(device)
  model.eval()
  return model

style_model = load_resnet34_classifier(style_checkpoint, len(style_class_names))
type_model = load_resnet34_classifier(type_checkpoint, len(type_class_names))

print("Both models loaded successfully.")

Both models loaded successfully.


## 6. Prediction Pipeline

### 6.1. Image Transform

Before an image can be passed into the trained models, it must be processed in the same way as during training and evaluation.

The image is resized to 224 by 224 pixels, converted into a tensor, and normalized using ImageNet statistics.

In [4]:
inference_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

### 6.2 Prediction Function

A helper function is defined to run inference using both the style and clothing type classifiers.

The function returns:

- predicted style
- predicted clothing type
- confidence score for each prediction

In [5]:
def predict_style_and_type(image_path):
    image = Image.open(image_path).convert("RGB")
    input_tensor = inference_transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        style_outputs = style_model(input_tensor)
        type_outputs = type_model(input_tensor)

        style_probs = torch.softmax(style_outputs, dim=1)
        type_probs = torch.softmax(type_outputs, dim=1)

        style_pred_idx = torch.argmax(style_probs, dim=1).item()
        type_pred_idx = torch.argmax(type_probs, dim=1).item()

        predicted_style = style_class_names[style_pred_idx]
        predicted_type = type_class_names[type_pred_idx]

        style_confidence = style_probs[0][style_pred_idx].item()
        type_confidence = type_probs[0][type_pred_idx].item()

    return {
        "style": predicted_style,
        "type": predicted_type,
        "style_confidence": style_confidence,
        "type_confidence": type_confidence
    }

### 6.3 Example Prediction

To verify that both models work correctly, a test image is passed through the prediction function.

In [9]:
test_image_path = "../dataset/cleaned/gothic/jacket/gothic_jacket_025.png"

prediction = predict_style_and_type(test_image_path)
prediction

{'style': 'gothic',
 'type': 'jacket',
 'style_confidence': 0.9474146366119385,
 'type_confidence': 0.8012428283691406}